# VoyageIQ — Total Trip Cost Forecasting
## Exploratory notebook sandbox

This notebook is an exploratory sandbox for Machine Learning Operations (MLOps) development

Use it to
- Inspect raw and cleaned data
- Perform focused Exploratory Data Analysis (EDA) checks to validate assumptions
- Run quick training and inference checks while iterating on features
- Experiment with new features and models to improve metrics performance

Do not use it to
- Replace the reproducible pipeline in `src/main.py`
- Store production logic in notebook cells
- Write production artifacts to disk from the notebook

Canonical production entry point
- Run from terminal: `python -m src.main`

Why this separation matters
- The orchestrator (`src/main.py`) is the factory: deterministic inputs, deterministic outputs, clear provenance
- The notebook is the lab bench: interactive inspection, rapid iteration, visible intermediate states and exploration

### A) Environment setup and imports

Learning intent
- Make imports reliable regardless of how you opened Jupyter
- Ensure relative paths behave the same way for everyone
- Keep notebook code thin by calling functions from `src/` modules

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import os # For changing the working directory to the repo root
import sys # For manipulating sys.path to ensure imports work regardless of where the notebook is started
from pathlib import Path # For working with file paths in a platform-independent way

import pandas as pd
from IPython.display import display # For displaying dataframes nicely in Jupyter

# Repo root alignment
# Problem this solves
# - Notebooks can start with different working directories depending on IDE and settings
# - Relative paths then break unpredictably
#
# Approach
# - Search upward from the current folder until we find a folder that contains `src/`
# - Set that as the working directory so relative paths match the orchestrator behaviour
def find_repo_root(start: Path, marker_dir: str = "src", max_hops: int = 12) -> Path:
    current = start.resolve()
    for _ in range(max_hops):
        if (current / marker_dir).exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    raise RuntimeError(
        f"Could not find repo root containing '{marker_dir}/' starting from: {start}"
    )

PROJECT_ROOT = find_repo_root(Path.cwd())
os.chdir(PROJECT_ROOT)

# Ensure `import src...` works even if Jupyter started elsewhere
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())

# Imports from production modules
# Note: we intentionally do NOT import from src/main.py
from src.load_data import load_raw_data
from src.clean_data import clean_dataframe
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor
from src.train import train_model
from src.evaluate import evaluate_model
from src.infer import run_inference

PROJECT_ROOT: C:\Users\moham\Desktop\Term 2\MLOPS\1st group project - claude\voyageiq_mlops_pipeline_V5
cwd: C:\Users\moham\Desktop\Term 2\MLOPS\1st group project - claude\voyageiq_mlops_pipeline_V5


### B) Sandbox configuration

Guideline
- Keep configuration in one place
- Use the same column names and defaults as the orchestrator where possible
- Prefer explicit lists over clever inference so errors are actionable

In [3]:
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "travel_raw.csv"

SETTINGS = {
    "target_column": "total_cost",
    "problem_type": "regression",  # "regression" or "classification"
    "evaluate_on_test": False,  # Protect the audit vault until final evaluation
    "split": {"test_size": 0.15, "val_size": 0.15, "random_state": 42},
    "features": {
        "quantile_bin": [],  # No binning for travel cost regression
        "categorical_onehot": [
            "destination_country",
            "traveler_gender",
            "traveler_nationality",
            "accommodation_type",
            "transportation_type",
        ],
        "numeric_passthrough": [
            "duration_days",
            "traveler_age",
            "travel_month",
            "day_of_week",
        ],
        "n_bins": 3,
    },
}

def three_way_split(
    X: pd.DataFrame,
    y: pd.Series,
    *,
    test_size: float,
    val_size: float,
    random_state: int,
    stratify: bool,
):
    """
    Sandbox copy aligned with the orchestrator intent
    - Students benefit from seeing the leakage gate explicitly
    - We avoid importing helpers from src/main.py to prevent side effects

    Behaviour
    - Try stratified splits for classification
    - If stratification fails, fall back to random splits with a clear message
    """
    from sklearn.model_selection import train_test_split

    if test_size <= 0 or val_size <= 0 or (test_size + val_size) >= 1.0:
        raise ValueError("Split sizes must satisfy 0 < test_size, 0 < val_size, and test_size + val_size < 1")

    stratify_y = y if stratify else None

    try:
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify_y,
        )

        relative_val_size = val_size / (1.0 - test_size)
        stratify_temp = y_temp if stratify else None

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp,
            test_size=relative_val_size,
            random_state=random_state,
            stratify=stratify_temp,
        )

        return X_train, X_val, X_test, y_train, y_val, y_test

    except ValueError as e:
        print(f"[notebook] Stratified split failed: {e}")
        print("[notebook] Falling back to random split")

        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=random_state,
        )

        relative_val_size = val_size / (1.0 - test_size)

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp,
            test_size=relative_val_size,
            random_state=random_state,
        )

        return X_train, X_val, X_test, y_train, y_val, y_test

print("RAW_DATA_PATH:", RAW_DATA_PATH)
print("TARGET:", SETTINGS["target_column"])
print("PROBLEM_TYPE:", SETTINGS["problem_type"])

RAW_DATA_PATH: C:\Users\moham\Desktop\Term 2\MLOPS\1st group project - claude\voyageiq_mlops_pipeline_V5\data\raw\travel_raw.csv
TARGET: total_cost
PROBLEM_TYPE: regression


### 1) Load raw data (`src.load_data`)

Educational note
- We load raw data exactly once
- Raw data should be treated as immutable input

In [4]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw data not found at {RAW_DATA_PATH}\n"
        "Check that you opened the correct repository folder and that data/raw exists"
    )

df_raw = load_raw_data(RAW_DATA_PATH)
print("df_raw.shape:", df_raw.shape)
df_raw.head()

[load_data] Loading raw data from: C:\Users\moham\Desktop\Term 2\MLOPS\1st group project - claude\voyageiq_mlops_pipeline_V5\data\raw\travel_raw.csv
[utils] Loading CSV from: C:\Users\moham\Desktop\Term 2\MLOPS\1st group project - claude\voyageiq_mlops_pipeline_V5\data\raw\travel_raw.csv
[utils] CSV loaded — shape: (139, 13)
[load_data] Data loaded successfully — shape: 139 rows x 13 columns
df_raw.shape: (139, 13)


,Trip ID,Destination,Start date,End date,Duration (days),Traveler name,Traveler age,Traveler gender,Traveler nationality,Accommodation type,Accommodation cost,Transportation type,Transportation cost
0,1,"London, UK",5/1/2023,5/8/2023,7.0,John Smith,35.0,Male,American,Hotel,1200,Flight,600
1,2,"Phuket, Thailand",6/15/2023,6/20/2023,5.0,Jane Doe,28.0,Female,Canadian,Resort,800,Flight,500
2,3,"Bali, Indonesia",7/1/2023,7/8/2023,7.0,David Lee,45.0,Male,Korean,Villa,1000,Flight,700
3,4,"New York, USA",8/15/2023,8/29/2023,14.0,Sarah Johnson,29.0,Female,British,Hotel,2000,Flight,1000
4,5,"Tokyo, Japan",9/10/2023,9/17/2023,7.0,Kim Nguyen,26.0,Female,Vietnamese,Airbnb,700,Train,200


### 2) Focused EDA checks

Goal
- Validate assumptions before investing in feature engineering and training

Guideline
- Prefer small, targeted checks over long notebooks
- If an assumption is wrong, fix the pipeline, not the notebook

In [5]:
print("Missing values (top 10 columns):")
display(df_raw.isna().sum().sort_values(ascending=False).head(10))

target_col = SETTINGS["target_column"]
print(f"\nNote: Target column '{target_col}' is created during cleaning")
print(f"(total_cost = accommodation_cost + transportation_cost)")

# Inspect the cost component columns in raw data
for cost_col in ["Accommodation cost", "Transportation cost"]:
    if cost_col in df_raw.columns:
        print(f"\nSummary for '{cost_col}':")
        display(df_raw[cost_col].describe())

# Check destination format
if "Destination" in df_raw.columns:
    print("\nSample destinations (City, Country format):")
    display(df_raw["Destination"].head(5))

# Check date format
if "Start date" in df_raw.columns:
    print("\nSample start dates:")
    display(df_raw["Start date"].head(5))

print(f"\nDataset shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")

Missing values (top 10 columns):


Transportation type     3
Transportation cost     3
Start date              2
End date                2
Destination             2
Traveler nationality    2
Duration (days)         2
Traveler name           2
Traveler age            2
Accommodation cost      2
dtype: int64


Note: Target column 'total_cost' is created during cleaning
(total_cost = accommodation_cost + transportation_cost)

Summary for 'Accommodation cost':


count      137
unique      53
top       1200
freq         7
Name: Accommodation cost, dtype: object


Summary for 'Transportation cost':


...


Sample destinations (City, Country format):


0          London, UK
1    Phuket, Thailand
2     Bali, Indonesia
3       New York, USA
4        Tokyo, Japan
Name: Destination, dtype: str


Sample start dates:


0     5/1/2023
1    6/15/2023
2     7/1/2023
3    8/15/2023
4    9/10/2023
Name: Start date, dtype: str


Dataset shape: 139 rows × 13 columns
Columns: ['Trip ID', 'Destination', 'Start date', 'End date', 'Duration (days)', 'Traveler name', 'Traveler age', 'Traveler gender', 'Traveler nationality', 'Accommodation type', 'Accommodation cost', 'Transportation type', 'Transportation cost']


### 3) Clean data (`src.clean_data`)

Educational note
- Cleaning should be deterministic
- Cleaning should not learn from the data in a way that leaks information across splits
- Our cleaning step: standardises column names, removes duplicates/NaN, parses currency values, creates `total_cost`, extracts `destination_country`, `travel_month`, `day_of_week`

In [6]:
df_clean = clean_dataframe(df_raw, target_column=SETTINGS["target_column"])
print("df_clean.shape:", df_clean.shape)
df_clean.head()

[clean_data] Cleaning started — initial rows: 139
[clean_data] Column names standardised: ['trip_id', 'destination', 'start_date', 'end_date', 'duration_days', 'traveler_name', 'traveler_age', 'traveler_gender', 'traveler_nationality', 'accommodation_type', 'accommodation_cost', 'transportation_type', 'transportation_cost']
[clean_data] Dropped columns (if present): ['trip_id', 'traveler_name']
[clean_data] Duplicates removed: 1
[clean_data] Rows dropped (NaN): 2
[clean_data] Target column 'total_cost' created.
[clean_data] Cleaning complete — final rows: 136 (dropped: 3)
df_clean.shape: (136, 16)


,destination,start_date,end_date,duration_days,traveler_age,traveler_gender,traveler_nationality,accommodation_type,accommodation_cost,transportation_type,transportation_cost,total_cost,destination_city,destination_country,travel_month,day_of_week
0,"London, UK",5/1/2023,5/8/2023,7.0,35.0,Male,American,Hotel,1200,Flight,600,1800,London,UK,5,0
1,"Phuket, Thailand",6/15/2023,6/20/2023,5.0,28.0,Female,Canadian,Resort,800,Flight,500,1300,Phuket,Thailand,6,3
2,"Bali, Indonesia",7/1/2023,7/8/2023,7.0,45.0,Male,Korean,Villa,1000,Flight,700,1700,Bali,Indonesia,7,5
3,"New York, USA",8/15/2023,8/29/2023,14.0,29.0,Female,British,Hotel,2000,Flight,1000,3000,New York,USA,8,1
4,"Tokyo, Japan",9/10/2023,9/17/2023,7.0,26.0,Female,Vietnamese,Airbnb,700,Train,200,900,Tokyo,Japan,9,6


### 4) Didactic check: what changed after cleaning

Goal
- Make transformations visible
- Help you debug unexpected column name changes or dropped columns

In [7]:
raw_cols = list(df_raw.columns)
clean_cols = list(df_clean.columns)

removed_cols = sorted(set(raw_cols) - set(clean_cols))
added_cols = sorted(set(clean_cols) - set(raw_cols))

print("Columns removed (raw names no longer present after standardisation + drop):")
display(removed_cols)

print("\nColumns added during cleaning:")
display(added_cols)

print('\nCleaned has "total_cost":', "total_cost" in df_clean.columns)
print('Cleaned has "destination_country":', "destination_country" in df_clean.columns)
print('Cleaned has "travel_month":', "travel_month" in df_clean.columns)

print(f"\nRows: {df_raw.shape[0]} raw → {df_clean.shape[0]} clean (dropped {df_raw.shape[0] - df_clean.shape[0]})")

Columns removed (raw names no longer present after standardisation + drop):


['Accommodation cost',
 'Accommodation type',
 'Destination',
 'Duration (days)',
 'End date',
 'Start date',
 'Transportation cost',
 'Transportation type',
 'Traveler age',
 'Traveler gender',
 'Traveler name',
 'Traveler nationality',
 'Trip ID']


Columns added during cleaning:


['accommodation_cost',
 'accommodation_type',
 'day_of_week',
 'destination',
 'destination_city',
 'destination_country',
 'duration_days',
 'end_date',
 'start_date',
 'total_cost',
 'transportation_cost',
 'transportation_type',
 'travel_month',
 'traveler_age',
 'traveler_gender',
 'traveler_nationality']


Cleaned has "total_cost": True
Cleaned has "destination_country": True
Cleaned has "travel_month": True

Rows: 139 raw → 136 clean (dropped 3)


### 5) Validate data (security gate)

Educational note
- Validation is a fail-fast gate
- It prevents wasting time on training with broken assumptions

Guideline
- If validation fails, fix the upstream module or the data contract
- Do not patch around failures inside the notebook

In [8]:
required_columns = (
    [SETTINGS["target_column"]]
    + SETTINGS["features"]["quantile_bin"]
    + SETTINGS["features"]["categorical_onehot"]
    + SETTINGS["features"]["numeric_passthrough"]
)
required_columns = list(dict.fromkeys(required_columns))  # deduplicate

validate_dataframe(
    df=df_clean,
    required_columns=required_columns,
)

print("[notebook] Validation passed")

[validate] Starting data validation …
[validate] Validation passed — no issues detected.
[notebook] Validation passed


### 6) Build the feature recipe (`src.features`)

Educational note
- This step builds a preprocessing blueprint
- It must not fit on the full dataset in the notebook
- The recipe learns only when fitted on the training split inside the training pipeline

In [9]:
preprocessor = get_feature_preprocessor(
    quantile_bin_cols=SETTINGS["features"]["quantile_bin"],
    categorical_onehot_cols=SETTINGS["features"]["categorical_onehot"],
    numeric_passthrough_cols=SETTINGS["features"]["numeric_passthrough"],
    n_bins=SETTINGS["features"]["n_bins"],
)

print("[notebook] Feature recipe built, not fitted yet")
preprocessor

[features] Building feature preprocessor recipe …
[features] Preprocessor recipe built — quantile_bin: [], passthrough: ['duration_days', 'traveler_age', 'travel_month', 'day_of_week'], categorical: ['destination_country', 'traveler_gender', 'traveler_nationality', 'accommodation_type', 'transportation_type']
[notebook] Feature recipe built, not fitted yet


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_passthrough', ...), ('cat_onehot', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

### 7) Leakage gate: three-way split (train, validation, test)

Educational note
- Train split is the only split allowed to learn preprocessing parameters and model weights
- Validation split is for iteration and model selection
- Test split is a final audit vault

In [10]:
# Determine feature columns (everything except the target)
feat_cfg = SETTINGS["features"]
keep_cols = (
    feat_cfg["numeric_passthrough"]
    + feat_cfg["categorical_onehot"]
    + feat_cfg["quantile_bin"]
)
keep_cols = [c for c in keep_cols if c in df_clean.columns]

X = df_clean[keep_cols]
y = df_clean[SETTINGS["target_column"]]

X_train, X_val, X_test, y_train, y_val, y_test = three_way_split(
    X,
    y,
    test_size=SETTINGS["split"]["test_size"],
    val_size=SETTINGS["split"]["val_size"],
    random_state=SETTINGS["split"]["random_state"],
    stratify=(SETTINGS["problem_type"] == "classification"),
)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

if len(X_test) == 0:
    raise ValueError("Test split is empty. Check split ratios and dataset size.")

Train: (94, 9) Validation: (21, 9) Test: (21, 9)


### 8) Train (`src.train`)

Educational note
- This is the only step where `.fit()` happens
- The preprocessor and estimator are fitted only on training data

In [11]:
model_pipeline = train_model(
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor,
    problem_type=SETTINGS["problem_type"],
)

print("[notebook] Training complete")
model_pipeline

[train] Training model — problem_type: regression
[train] Fitting pipeline on 94 training samples …
[train] Training complete.
[notebook] Training complete


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_passthrough', ...), ('cat_onehot', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differen

### 9) Evaluate (`src.evaluate`)

**Educational Note: The Audit Vault**
* **Validation:** Used to iteratively guide our model and feature engineering decisions.
* **Test:** Our blind audit vault. We use a strict boolean flag (`evaluate_on_test` in our `SETTINGS`) to ensure we only peek at this vault when we are absolutely ready to finalise the model.

**Metric Note**
* `evaluate_model` returns a single float based on `problem_type`.
* **Regression:** Returns **RMSE** (Root Mean Squared Error) — also prints MAE and R² for observability.
* **Classification:** Returns **F1** (weighted) for multiclass support.

In [12]:
val_metric = evaluate_model(
    model=model_pipeline,
    X_test=X_val,
    y_test=y_val,
    problem_type=SETTINGS["problem_type"],
)
print(f"[notebook] Validation metric: {val_metric:.4f}")

# The "Break Glass" Audit Step
if SETTINGS.get("evaluate_on_test", False):
    test_metric = evaluate_model(
        model=model_pipeline,
        X_test=X_test,
        y_test=y_test,
        problem_type=SETTINGS["problem_type"],
    )
    print(f"[notebook] Test metric (audit vault): {test_metric:.4f}")
else:
    print("[notebook] Test metrics: Skipped to protect the audit vault. Set 'evaluate_on_test': True to run.")

metric_name = "RMSE" if SETTINGS["problem_type"] == "regression" else "F1 (weighted)"
print(f"\n💡 Note: evaluate_model returns {metric_name} because problem_type='{SETTINGS['problem_type']}'.")

[evaluate] Evaluating model — problem_type: regression
[evaluate] RMSE: 1998.8548 | MAE: 1087.5988 | R2: 0.1744
[notebook] Validation metric: 1998.8548
[notebook] Test metrics: Skipped to protect the audit vault. Set 'evaluate_on_test': True to run.

💡 Note: evaluate_model returns RMSE because problem_type='regression'.


### 10) Inference demo (`src.infer`)

Educational note
- Inference simulates what happens after training
- We deliberately use rows from the test split to simulate unseen cases

In [13]:
sample_n = min(10, len(X_test))
X_infer_sample = X_test.sample(n=sample_n, random_state=SETTINGS["split"]["random_state"])

df_predictions = run_inference(
    model=model_pipeline,
    X_infer=X_infer_sample,
)

print("[notebook] Inference results")
display(df_predictions.head(10))

[infer] Running inference on 10 samples …
[infer] Inference complete — 10 predictions generated.
[notebook] Inference results


,prediction
73,3259.112554
105,2113.098972
40,1629.825754
45,3620.817713
19,2148.673882
62,1245.951206
51,1367.262024
42,2091.965491
4,1196.411513
132,4450.255355


### 11) Inspect production artifacts produced by the orchestrator

This notebook does not write artifacts to disk

Use this cell only after you run the orchestrator from terminal
- `python -m src.main`

Goal
- Prove that the factory output exists on disk
- Inspect outputs without modifying them

In [14]:
from src.utils import load_model

CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "model.joblib"
PREDICTIONS_PATH = PROJECT_ROOT / "reports" / "predictions.csv"

try:
    clean_from_disk = pd.read_csv(CLEAN_DATA_PATH)
    preds_from_disk = pd.read_csv(PREDICTIONS_PATH)
    model_from_disk = load_model(MODEL_PATH)

    print("clean.csv shape:", clean_from_disk.shape)
    print("predictions.csv shape:", preds_from_disk.shape)
    print("loaded model type:", type(model_from_disk))

    display(preds_from_disk.head(10))

except Exception as e:
    print("Artifacts not found yet or could not be loaded")
    print("Run from terminal: python -m src.main")
    print("Error:", e)

Artifacts not found yet or could not be loaded
Run from terminal: python -m src.main
Error: [Errno 2] No such file or directory: 'C:\\Users\\moham\\Desktop\\Term 2\\MLOPS\\1st group project - claude\\voyageiq_mlops_pipeline_V5\\data\\processed\\clean.csv'
